# ASR-course Finetuning code


This notebook is a replication of the code used by the Whisper Finetuning event. Available here: https://github.com/huggingface/community-events/tree/main/whisper-fine-tuning-event.
Our approach is based on the paper by Jain et al. [1] Available here: https://doi.org/10.1109/ACCESS.2024.3378738
It is only slightly adapted to work with our hardware and datasets.
All comments in this notebook are there to explain the highlight the changes we made to the original notebook.

We made sure it only used one GPU on Ponyland, as it would otherwise result in many issues. This is not needed if you are not using Ponyland.

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [ ]:
from huggingface_hub import login
login()

Datasets are loaded in from Ponyland. If you are not using Ponyland, you need to change the code such that it takes the correct path to the dataset location.

In [ ]:
from pathlib import Path
import random
from sklearn.model_selection import train_test_split

In [ ]:
from datasets import load_from_disk 
train_dataset = load_from_disk("TLT_speed_train_dataset") 
val_dataset = load_from_disk("TLT_val_dataset_fix") 
Myst_test_dataset = load_from_disk("Myst_test_dataset") 
CMU_test_dataset = load_from_disk("CMU_test_dataset") 
TLT_test_dataset = load_from_disk("TLT_test_dataset")

In [ ]:
train_dataset = train_dataset.remove_columns("input_length")
eval_dataset = val_dataset.remove_columns("input_length")
Myst_test_dataset = Myst_test_dataset.remove_columns("input_length")
CMU_test_dataset = CMU_test_dataset.remove_columns("input_length")
TLT_test_dataset = TLT_test_dataset.remove_columns("input_length")

In [ ]:
from transformers import WhisperFeatureExtractor

feature_extractor = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")

In [ ]:
from transformers import WhisperTokenizer

tokenizer = WhisperTokenizer.from_pretrained("openai/whisper-small")

In [ ]:
from transformers import WhisperProcessor

processor = WhisperProcessor.from_pretrained("openai/whisper-small")

In [ ]:
from transformers.models.whisper.english_normalizer import BasicTextNormalizer

do_lower_case = False
do_remove_punctuation = False

normalizer = BasicTextNormalizer()

In [ ]:
import torch

from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # split inputs and labels since they have to be of different lengths and need different padding methods
        # first treat the audio inputs by simply returning torch tensors
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # get the tokenized label sequences
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        # pad the labels to max length
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # replace padding with -100 to ignore loss correctly
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # if bos token is appended in previous tokenization step,
        # cut bos token here as it's append later anyways
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        return batch

In [ ]:
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

We added Cer metric and substitution, deletion and insertion rate in the compute metrics function

In [ ]:
import evaluate

metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

In [ ]:
from jiwer import process_words
# evaluate with the 'normalised' WER
do_normalize_eval = True

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids

    # replace -100 with the pad_token_id
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id

    # we do not want to group tokens when computing the metrics
    pred_str = processor.tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.tokenizer.batch_decode(label_ids, skip_special_tokens=True)

    if do_normalize_eval:
        pred_str = [normalizer(pred) for pred in pred_str]
        label_str = [normalizer(label) for label in label_str]

    p_words = process_words(label_str, pred_str)


    wer = 100 * metric.compute(predictions=pred_str, references=label_str)

    cer = 100 * cer_metric.compute(predictions=pred_str,references=label_str)

    total_words = sum(len(ref.split()) for ref in label_str)

    return {
    "wer": wer,
    "cer": cer,
    "sub_rate": 100 * p_words.substitutions / total_words,
    "del_rate": 100 * p_words.deletions / total_words,
    "ins_rate": 100 * p_words.insertions / total_words,
}

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

In [ ]:
# use to load our pretrained models from huggingface
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained(
    "Kwimp/TLT_spec_speed_augment_whisper_small"
)

We added SpecAugment here, to be able to use it during finetuning

In [ ]:
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []
model.config.use_cache = False
model.generation_config.language = "en"
model.generation_config.task = "transcribe"
model.config.apply_spec_augment = True

In [ ]:
print(model.config.mask_time_prob)
print(model.config.mask_time_length)
print(model.config.mask_feature_prob)
print(model.config.mask_feature_length)
print(model.config.apply_spec_augment)


In [ ]:
model.proj_out

We freezed every layer except the final layer to match the source paper

In [ ]:
for param in model.parameters():
    param.requires_grad = False

for param in model.proj_out.parameters():
    param.requires_grad = True

Training arguments are adapted to match the source paper

In [ ]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="TLT_spec_augment_speed_whisper_small",
    per_device_train_batch_size=32,
    gradient_accumulation_steps=2,  # increase by 2x for every 2x decrease in batch size
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    learning_rate=1e-5,
    warmup_steps=2048,
    max_steps=4000,
    gradient_checkpointing=True, # original set to True
    fp16=True,
    eval_strategy="steps",
    per_device_eval_batch_size=8,
    predict_with_generate=True, # original set to True
    generation_max_length=225,
    save_steps=500,
    eval_steps=500,
    logging_steps=25,
    report_to=["tensorboard"],
    load_best_model_at_end=True,
    metric_for_best_model="wer",
    greater_is_better=False,
    push_to_hub=True, # set to true if we plan to use huggingface hub
)

In [ ]:
from transformers import Seq2SeqTrainer

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    processing_class=processor,
)

In [ ]:
processor.save_pretrained(training_args.output_dir)

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
print(next(model.parameters()).device)

In [ ]:
trainer.train()

We can label our checkpoint with the `whisper-event` tag on push by setting the appropriate key-word arguments (kwargs):

In [ ]:
kwargs = {
    "dataset": "LTL2021",  # a 'pretty' name for the training dataset
    "language": "en",
    "model_name": "whisper small finetuned Specaug whole data, speed non-native",  # a 'pretty' name for your model
    "finetuned_from": "openai/whisper-small",
    "tasks": "automatic-speech-recognition",
}

The training results can now be uploaded to the Hub. To do so, execute the `push_to_hub` command and save the preprocessor object we created:

In [ ]:
trainer.push_to_hub(**kwargs)

In [ ]:
We added the evaluation for the three datasets.

In [ ]:
trainer.evaluate(Myst_test_dataset)

In [ ]:
trainer.evaluate(CMU_test_dataset)

In [ ]:
trainer.evaluate(TLT_test_dataset)

## References

[1] Jain, Rishabh & Barcovschi, Andrei & Yiwere, Mariam & Corcoran, Peter & Cucu, Horia. (2024). Exploring Native and Non-Native English Child Speech Recognition With Whisper. IEEE Access. PP. 1-1. 10.1109/ACCESS.2024.3378738.